In [2]:
import pandas as pd
data = pd.read_csv(r"D:\ansci-4040-fall-2026\jl4937_6040-project-1\dataset\Data_set_prep_assignment_1.csv", low_memory=False)


In [3]:
#Find the number of duplicate rows in the dataset
duplicate_count = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 60653


In [4]:
# Remove duplicate rows while keeping the first (original) occurrence
original_rows = len(data)

data = data.drop_duplicates(keep="first").reset_index(drop=True)

print(f"Removed {original_rows - len(data):,} duplicate rows.")
print(f"Remaining rows: {len(data):,}")

Removed 60,653 duplicate rows.
Remaining rows: 8,434,768


In [5]:
# Check for duplicate rows after removing duplicates
remaining_duplicates = data.duplicated().sum()

print(f"Remaining duplicate rows: {remaining_duplicates:,}")
print("Duplicate rows still exist." if remaining_duplicates > 0 else "No duplicate rows remain.")

Remaining duplicate rows: 0
No duplicate rows remain.


In [6]:
# Count null values in each variable
null_counts = data.isna().sum().sort_values(ascending=False)

print(null_counts)

DaysInMilk                1696874
AnimalId                  1696870
LactationNumber           1696870
ReproductionStatus        1696870
Avgmilkflow                   172
EventDate                       0
Flow30_60Session                0
YieldFirst2Min_Session          0
YieldSession                    0
DurationSession_sec             0
milking                         0
dtype: int64


In [7]:
# Remove rows with null Avgmilkflow values
before = len(data)

data_clean = data.dropna(subset=["Avgmilkflow"]).reset_index(drop=True)

print(f"Removed {before - len(data_clean):,} rows with null Avgmilkflow values.")
print(f"Remaining rows: {len(data_clean):,}")

Removed 172 rows with null Avgmilkflow values.
Remaining rows: 8,434,596


In [8]:
# Count zero values in each variable
zero_counts = data.eq(0).sum().sort_values(ascending=False)

print("Zero values by column:")
print(zero_counts)

Zero values by column:
Flow30_60Session          3139
Avgmilkflow                204
AnimalId                     0
DaysInMilk                   0
LactationNumber              0
EventDate                    0
ReproductionStatus           0
YieldFirst2Min_Session       0
YieldSession                 0
DurationSession_sec          0
milking                      0
dtype: int64


In [9]:
# Remove rows with zero Avgmilkflow values
zero_mask = data["Avgmilkflow"].eq(0)

print(f"Rows removed: {zero_mask.sum():,}")

data = data.loc[~zero_mask].reset_index(drop=True)

print(f"Remaining rows: {len(data):,}")

Rows removed: 204
Remaining rows: 8,434,564


In [10]:
# Count zero values in each variable
zero_counts = (data == 0).sum().sort_values(ascending=False)

print("Zero values by column:")
print(zero_counts)

Zero values by column:
Flow30_60Session          3139
LactationNumber              0
AnimalId                     0
DaysInMilk                   0
ReproductionStatus           0
EventDate                    0
Avgmilkflow                  0
YieldFirst2Min_Session       0
YieldSession                 0
DurationSession_sec          0
milking                      0
dtype: int64


In [11]:
# Check null values in the current data DataFrame
null_counts = data.isna().sum().sort_values(ascending=False)

print("Null values by column:")
print(null_counts)

print(f"\nTotal null values: {data.isna().sum().sum():,}")

Null values by column:
DaysInMilk                1696834
AnimalId                  1696830
LactationNumber           1696830
ReproductionStatus        1696830
Avgmilkflow                   172
EventDate                       0
Flow30_60Session                0
YieldFirst2Min_Session          0
YieldSession                    0
DurationSession_sec             0
milking                         0
dtype: int64

Total null values: 6,787,496


In [12]:
# Columns representing production or milking measurements
production_cols = [
    "Avgmilkflow",  
]

# Step 1: remove rows containing zero OR null production values
zero_mask = data[production_cols].eq(0).any(axis=1)
null_mask = data[production_cols].isnull().any(axis=1)

print(f"Rows removed because of zero values: {zero_mask.sum():,}")
print(f"Rows removed because of null values: {null_mask.sum():,}")

data_clean = data.loc[~zero_mask & ~null_mask].copy()

Rows removed because of zero values: 0
Rows removed because of null values: 172


In [13]:
# Count zero values in each variable
zero_counts = (data == 0).sum().sort_values(ascending=False)

print(zero_counts)

Flow30_60Session          3139
LactationNumber              0
AnimalId                     0
DaysInMilk                   0
ReproductionStatus           0
EventDate                    0
Avgmilkflow                  0
YieldFirst2Min_Session       0
YieldSession                 0
DurationSession_sec          0
milking                      0
dtype: int64


In [14]:
# Continuous milking-performance variables used for Z-score filtering
zscore_cols = [
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
]

# Calculate Z-scores
z_scores = (
    data_clean[zscore_cols] - data_clean[zscore_cols].mean()
) / data_clean[zscore_cols].std(ddof=0)

# Identify rows with at least one extreme Z-score
outlier_mask = z_scores.abs().gt(3).any(axis=1)

outliers_by_variable = z_scores.abs().gt(3).sum()

print("Outliers detected by variable:")
print(outliers_by_variable)

# Remove identified outliers
data_zfiltered = data_clean.loc[~outlier_mask].reset_index(drop=True)

print(f"Rows removed by Z-score filtering: {outlier_mask.sum():,}")
print(f"Remaining rows: {len(data_zfiltered):,}")

Outliers detected by variable:
Avgmilkflow                3599
Flow30_60Session            839
YieldFirst2Min_Session        0
YieldSession              20622
DurationSession_sec       31948
dtype: int64
Rows removed by Z-score filtering: 53,775
Remaining rows: 8,380,617


In [15]:
# Expert-informed plausibility limits for dairy milking records.
# Limits are intentionally conservative but broad enough to retain normal cows.

thresholds = {
    "LactationNumber": (1, 10),
    "DaysInMilk": (0, 500),
    "Avgmilkflow": (0.2, 8.0),               # kg/min
    "Flow30_60Session": (0.1, 10.0),         # kg/min
    "YieldFirst2Min_Session": (0.1, 15.0),   # kg
    "YieldSession": (0.5, 70.0),             # kg
    "DurationSession_sec": (60, 900),        # seconds
}

# Use the already Z-score-filtered dataset
data_thresholdfiltered = data_zfiltered.copy()

# Remove rows with missing values in variables required for threshold filtering
required_cols = list(thresholds)
missing_mask = data_thresholdfiltered[required_cols].isna().any(axis=1)

# Apply all lower and upper cutoffs
outside_range_mask = pd.Series(False, index=data_thresholdfiltered.index)

for column, (lower, upper) in thresholds.items():
    outside_range_mask |= ~data_thresholdfiltered[column].between(
        lower, upper, inclusive="both"
    )

remove_mask = missing_mask | outside_range_mask

data_thresholdfiltered = (
    data_thresholdfiltered.loc[~remove_mask]
    .reset_index(drop=True)
)

print(f"Rows removed for missing threshold variables: {missing_mask.sum():,}")
print(f"Rows removed for implausible values: {outside_range_mask.sum():,}")
print(f"Total rows removed: {remove_mask.sum():,}")
print(f"Remaining rows: {len(data_thresholdfiltered):,}")

Rows removed for missing threshold variables: 1,685,832
Rows removed for implausible values: 1,735,445
Total rows removed: 1,735,445
Remaining rows: 6,645,172


In [17]:
from sklearn.model_selection import train_test_split

# Set features and target
X = data_thresholdfiltered.drop(columns=["AnimalId"])
y = data_thresholdfiltered["AnimalId"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (5316137, 10)
X_test shape: (1329035, 10)
y_train shape: (5316137,)
y_test shape: (1329035,)
